# Results for model: moonshotai/kimi-k2-instruct-0905

In [ ]:
import polars as pl
import xgboost as xgb
from pathlib import Path

data_path = Path('./jane_street_data/train.parquet')
df = pl.scan_parquet(data_path)

TOP_QUANTILE = 0.85

df_feat = (
    df
    .with_columns(
        pl.col('feature_00').quantile(TOP_QUANTILE).over('date_id').alias('feat00_q85')
    )
    .with_columns(
        (pl.col('feature_00') - pl.col('feat00_q85')).alias('feat00_q85_diff')
    )
    .select(
        pl.col('feature_00'),
        pl.col('feat00_q85_diff'),
        pl.col('responder_6').alias('target')
    )
    .drop_nulls()
    .collect()
)

X = df_feat[['feature_00', 'feat00_q85_diff']].to_numpy()
y = df_feat['target'].to_numpy()

model = xgb.XGBRegressor(
    n_estimators=800,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    objective='reg:squarederror',
    n_jobs=-1
)

model.fit(X, y)